In [0]:
%sql
describe customers;

In [0]:
%sql
describe orders;

In [0]:
%sql
describe order_details;

#### Sub-Query

In [0]:
%sql
-- Find customers who placed orders above the average order amount.

select distinct c.first_name, c.last_name, o.total_amount
from customers c
inner join orders o
on c.customer_id = o.customer_id
where total_amount > (select avg(total_amount) from orders);

In [0]:
%sql
-- List all products that have been sold more than 2 times in total.

select od.product_name, sum(od.quantity) as total_sold
from order_details od
group by product_name
having sum(od.quantity) > (select 2);

#### Common Table Expressions (CTEs)

In [0]:
%sql
--  List customers and their total quantity of products ordered.

with customer_orders as (select c.first_name, c.last_name, sum(od.quantity) as total_quantity
from customers c
inner join orders o on c.customer_id = o.customer_id
inner join order_details od on o.order_id = od.order_id
group by c.customer_id, c.first_name, c.last_name)

select * from customer_orders where total_quantity > 3;

In [0]:
%sql
--Find products with their total sales revenue (price * quantity).

WITH product_sales AS (select od.product_name, sum(od.price_each*od.quantity) as total_revenue
from order_details od
group by od.product_name)

select * from product_sales where total_revenue > 10000;

#### CASE Statements (SQL IF/ELSE)

In [0]:
%sql
--Categorize each order as 'Small', 'Medium', 'Large'.

WITH categorised_orders AS
(SELECT o.order_id, o.total_amount,
  CASE 
    WHEN o.total_amount < 1000 THEN 'Small'
    WHEN o.total_amount BETWEEN 1000 AND 2500 THEN 'Medium'
    ELSE 'Large'
  END AS order_size
FROM orders o)

SELECT * FROM categorised_orders
WHERE order_size = 'Large';

In [0]:
%sql
--Label customers as 'New' or 'Returning' based on order count.

SELECT c.first_name, c.last_name, COUNT(o.order_id) AS order_count,
  CASE 
    WHEN COUNT(o.order_id) = 1 THEN 'New'
    WHEN COUNT(o.order_id) > 1 THEN 'Returning'
    ELSE 'No Orders' 
  END AS customer_type
  FROM customers c
  LEFT JOIN orders o ON c.customer_id = o.customer_id
  GROUP BY c.first_name, c.last_name; 


#### FUNCTIONS (STRING + DATE)

In [0]:
%sql
SELECT UPPER(c.first_name) AS FIRST_NAME_UPPERCASE FROM customers c;

In [0]:
%sql
SELECT LOWER(c.first_name) AS FIRST_NAME_LOWERCASE FROM customers c;

In [0]:
%sql
SELECT UPPER(CONCAT(c.first_name, ' ', c.last_name)) AS FULL_NAME FROM customers c

In [0]:
%sql
-- Extract month of each order & group orders by month.

SELECT MONTH(o.order_date) AS order_month, COUNT(o.order_id) AS total_orders
FROM orders o  
GROUP BY 1
ORDER BY 1;

In [0]:
%sql
-- List each customer's name + their total spend + spending category.

SELECT UPPER(CONCAT(c.first_name, ' ', c.last_name)) AS customer_name, SUM(od.price_each*od.quantity) AS total_spend,
  CASE 
    WHEN total_spend < 10000 THEN 'Bronze'
    WHEN total_spend BETWEEN 10000 AND 20000 THEN 'Silver'
    ELSE 'Gold'
  END AS spending_category
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
INNER JOIN order_details od ON o.order_id = od.order_id
GROUP BY customer_name
ORDER BY total_spend DESC;